# 1. Import the necessary libraries

In [ ]:
from mesh.core import *
from problems import get_problem
from tuners import fine_tune_mesh

from pymoo.core.problem import Problem
from pymoo.optimize import minimize
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PolynomialMutation
from pymoo.indicators.hv import HV

import contextlib
import io
import optuna

# 2. Fine tuning configuration by hypervolume

In [ ]:
#################### CUSTOMIZABLE ####################

# Number of processes to execute the experiments in parallel
workers = 6

# Objective function configuration (function name, number of objectives, number of decision variables)
experiments = [('zdt1', 2, 10), ('dtlz1', 3, 10)]

# Algorithm configuration
population_size = 100 # Population size
max_iteration = None # Number of iterations/generations
max_fitness_eval = 1500 # Maximum number of fitness evaluations
random_state = None # Set a seed for random numbers (not used if it is None)

tuning_folder = f'hyperparams' # Folder to get the tuned parameters

# Fine tuning configuration
n_trials = 10 # Number of trials for Optuna
n_steps = 10 # Number of rounds in a trial
optuna_pruner = optuna.pruners.WilcoxonPruner(p_threshold=0.05, n_startup_steps=n_steps//2) # Optuna's pruner to prune bad trials according to a metric
hypercube_vertex = 100 # Consider the vertex of the hypercube
######################################################

ref_point = [np.array([hypercube_vertex] * exp[1]) for exp in experiments] # Reference point for hypervolume calculation
indicators = [HV(ref_point=rf) for rf in ref_point] # Define the indicator to calculate the volume

# 3 Define auxiliar functions

In [ ]:
def get_fitness_function(func_name, objective_dim, position_dim):
	func, position_min_value, position_max_value = get_problem(func_name, n_var=position_dim, n_obj=objective_dim)
	return func, objective_dim, position_dim, position_min_value, position_max_value

# Function to capture errors during parallel execution without stopping the other processes
def safe_run(run_function, params):
	try:
		f = io.StringIO()
		with contextlib.redirect_stdout(f):
			status = run_function(*params)
		return status, f.getvalue()
	except Exception as e:
		return f'Error: {str(e)}'

# 3. Apply fine tuning in the algorithms (in parallel)

## 3.1 Tuning MESH

In [ ]:
#################### CUSTOMIZABLE ####################

# MESH fixed parameters
mesh_memory_size = population_size # Maximum number of particles in memory
mesh_global_best_type = [0] # 0 -> Sigma method (G1) | 1 -> Sigma method in fronts (G2)
mesh_dm_pool_type = [0] # 0 -> Sampling from memory (S1) | 1 -> Sampling from population (S2) | 2 -> Sampling from memory and population (S3)
mesh_differential_evolution_type = [0] # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)
######################################################

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
     (n_trials, n_steps, optuna_pruner),
	 get_fitness_function(exp[0], exp[1], exp[2]),
	 (mesh_memory_size, gb_type, pool_type, de_type),
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(fine_tune_mesh, params) for params in params_list)

## Tuning MESH (previous implementation)

## 3.2 Tuning NSGA2

In [ ]:
class MyProblem(Problem):
    def __init__(self, n_var, xl, xu):
        super().__init__(n_var=n_var, n_obj=objective_dim, n_constr=0, xl=xl, xu=xu)

    def _evaluate(self, X, out, *args, **kwargs):
        out["F"] = np.array([func(x) for x in X])

problem = MyProblem(n_var=position_dim, xl=position_min_value, xu=position_max_value)

# Function used by hyperopt to optimize MESH for a function by the hypervolume
def optimize_hyperparameter_by_hypervolume_mesh(problem, hyperparameters, indicator):
    # Get the hyperparameters from the optimizer
	prob_rec = hyperparameters['recombination_probability']
	eta_rec = hyperparameters['eta_recombination']
	prob_mut = hyperparameters['mutation_probability']
	eta_mut = hyperparameters['eta_mutation']
	crossover = SBX(prob=prob_rec, prob_var=1.0, eta=eta_rec)
	mutation = PolynomialMutation(prob=prob_mut, eta=eta_mut)
    # Configure the NSGA-II algorithm
	nsga2 = NSGA2(
		pop_size=population_size,
		crossover=crossover,
		mutation=mutation,
		eliminate_duplicates=True,
		seed=random_state
	)
	# Run NSGA-II
	res = minimize(problem,
				nsga2,
                ('n_eval', max_fit_eval),
                verbose=False)
    # Calculate the hypervolume
	hypervolume = indicator(res.F)
	return -hypervolume

# Definition of design space of hyperparameters
hyperperameter_space = {
	"recombination_probability": hp.uniform("recombination_probability", 0, 1),
	"eta_recombination": hp.uniform("eta_recombination", 0, 20),
	"mutation_probability": hp.uniform("mutation_probability", 0, 1),
	"eta_mutation": hp.uniform("eta_mutation", 0, 20),
}

# Define the indicator to calculate the volume
indicator = HV(ref_point=ref_point)

best = fmin(lambda params: optimize_hyperparameter_by_hypervolume_mesh(problem, params, indicator), hyperperameter_space, algo=tpe.suggest, max_evals=n_trials, verbose=0)
for hyperparam, value in best.items():
    print(f"{hyperparam} = {value}")

## 3.3 Tuning ...